# Maintainer stage-gate — tilearray 0.2

**Purpose:** Drive the library yourself before cutting **0.2.0**. Checkout this PR branch, install editable, tweak the config cells, run against **live** public hosts, and judge API feel / progress / performance / small bugs.

**Not** Sphinx docs and **not** CI. Leave notes in the checklist at the bottom.

**Habit under test:** `config / preset → create_array → compute_with_policy`  
(`load_array` is eager sugar only — mention, don't use as the primary path.)

**Hosts:** OpenStreetMap XYZ (`for_openstreetmap`) and EA DSP Lidar WCS (`for_ea_dsp`).


## How to run this notebook

From the repo root (after checking out this PR branch):

```bash
uv sync --dev          # or: python -m venv .venv && source .venv/bin/activate && pip install -e ".[dev]"
uv run jupyter lab     # or: jupyter notebook
```

Open `notebooks/stage_gate_0_2.ipynb`. Edit the **config** cells, then Run All (or step through).

matplotlib + jupyter are already in the `[dev]` extra — no extra stack needed.


## Config — edit these knobs

Start **small**. Optional larger AOIs are below. Politeness: prefer preset ceilings; a lower `max_concurrent_requests` becomes call-local (`call_max_concurrent`) and does **not** shrink the shared host gate.


In [ ]:
from __future__ import annotations

import time
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

import tilearray
from tilearray import (
    WCSConfig,
    XYZConfig,
    compute_with_policy,
    create_array,
)
from tilearray.types import CRS, Format

print("tilearray", tilearray.__version__)

# --- AOI / zoom (edit me) -------------------------------------------------
# Small central-London window (default). Slightly larger optional AOI further down.
OSM_BBOX_SMALL = (-0.005, 51.495, 0.005, 51.505)  # lon/lat
OSM_BBOX_LARGER = (-0.02, 51.48, 0.02, 51.52)      # optional — more tiles, slower
OSM_ZOOM = 16
USE_LARGER_OSM_AOI = False

# EA Lidar — small OSGB window (~256 m). Optional larger ~800 m window.
EA_BBOX_SMALL = (431900.0, 382700.0, 432156.0, 382956.0)
EA_BBOX_LARGER = (431900.0, 382700.0, 432700.0, 383500.0)
EA_CHUNK = (64, 64)  # smaller chunks → more tiles (better politeness feel)
USE_LARGER_EA_AOI = False

# --- Politeness / fetch knobs (None = leave preset defaults) --------------
# OSM preset ceiling is 2; EA AIMD ceiling is 10. Lower values stay call-local.
OSM_MAX_CONCURRENT = None  # e.g. 1 to feel serial vs preset 2
EA_MAX_CONCURRENT = None   # e.g. 4 for call-local politeness under ceiling 10

# --- Soft-fail ------------------------------------------------------------
# Primary mosaics stay hard-fail (default). Soft-fail section toggles this.
SOFT_FAIL = True
SOFT_FAIL_MAX_FRACTION = 1.0  # allow all tiles to soft-fill in the deliberate-404 demo
SOFT_FAIL_FETCH_RETRIES = 0   # fail fast in the demo (don't wait on retries)

# --- Private / keyed servers (compose only — no AuthClient) ---------------
# Leave empty for public hosts. For private/keyed endpoints, set on the config:
#   params={"key": "..."} or {"username": "...", "password": "..."}
#   headers={"Authorization": "Bearer ..."}  # or Basic / X-API-Key
# Secrets from env, never committed. Caps↔tile credential parity: WCS/WMTS caps
# now inherit config headers/params (post-#57); prefer params/headers over URL secrets.
EXTRA_PARAMS: dict = {}
EXTRA_HEADERS: dict = {}


In [ ]:
def summarize(arr, *, label: str, wall_s: float | None = None) -> None:
    """Print shape/dtype/CRS + fetch-policy / soft-fail attrs."""
    print(f"=== {label} ===")
    if wall_s is not None:
        print(f"wall_time_s: {wall_s:.2f}")
    print(f"shape={arr.shape}  dtype={arr.dtype}  dims={arr.dims}")
    for key in ("crs", "service_type", "transform", "transform_source", "band_policy", "nodata"):
        if key in arr.attrs:
            print(f"  attrs[{key!r}] = {arr.attrs[key]!r}")
    fp = arr.attrs.get("tilearray_fetch_policy")
    if fp:
        print("  tilearray_fetch_policy:")
        pprint({k: fp[k] for k in (
            "max_concurrent", "call_max_concurrent", "adaptive_concurrency",
            "rate_limit_per_second", "retries", "initial_concurrent", "min_concurrent",
        ) if k in fp})
    te = arr.attrs.get("tilearray_tile_errors")
    if te is not None:
        print("  tilearray_tile_errors:")
        pprint(te)
    print()


def osm_bbox():
    return OSM_BBOX_LARGER if USE_LARGER_OSM_AOI else OSM_BBOX_SMALL


def ea_bbox():
    return EA_BBOX_LARGER if USE_LARGER_EA_AOI else EA_BBOX_SMALL


def _with_extras(cfg):
    """Merge EXTRA_PARAMS / EXTRA_HEADERS onto a preset config (keep OSM User-Agent)."""
    update = {}
    if EXTRA_PARAMS:
        update["params"] = {**(cfg.params or {}), **EXTRA_PARAMS}
    if EXTRA_HEADERS:
        update["headers"] = {**(cfg.headers or {}), **EXTRA_HEADERS}
    return cfg.model_copy(update=update) if update else cfg


def osm_config(**overrides):
    kwargs = {}
    if OSM_MAX_CONCURRENT is not None:
        kwargs["max_concurrent_requests"] = OSM_MAX_CONCURRENT
    kwargs.update(overrides)
    # Do not pass headers/params via kwargs — preset owns User-Agent; merge after.
    kwargs.pop("headers", None)
    kwargs.pop("params", None)
    return _with_extras(
        XYZConfig.for_openstreetmap(zoom=OSM_ZOOM, output_format=Format.PNG, **kwargs)
    )


def ea_config(**overrides):
    kwargs = {
        "crs": CRS.EPSG_27700,
        "output_format": Format.GEOTIFF,
        "chunk_size": EA_CHUNK,
        "resolution": (1.0, 1.0),
    }
    if EA_MAX_CONCURRENT is not None:
        kwargs["max_concurrent_requests"] = EA_MAX_CONCURRENT
    kwargs.update(overrides)
    kwargs.pop("headers", None)
    kwargs.pop("params", None)
    return _with_extras(
        WCSConfig.for_ea_dsp(
            "https://environment.data.gov.uk/spatialdata/"
            "lidar-composite-digital-terrain-model-dtm-1m/wcs",
            coverage_id=(
                "13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m"
            ),
            **kwargs,
        )
    )


## 1. Small polite mosaic + quick look (OSM XYZ)

Preset carries identifiable User-Agent and polite limits (2 in-flight, ~2 req/s). Include © OpenStreetMap contributors if you publish plots.

Palette PNGs may decode as **single-band** — that's expected today.


In [ ]:
cfg = osm_config()
da = create_array(cfg, osm_bbox(), CRS.EPSG_4326)
print("lazy:", da.shape, "chunks~", getattr(da.data, "chunks", None))

t0 = time.perf_counter()
arr = compute_with_policy(da)
wall = time.perf_counter() - t0
summarize(arr, label="OSM small mosaic", wall_s=wall)

plt.figure(figsize=(6, 4))
if "band" in arr.dims and arr.sizes["band"] >= 3:
    rgb = arr.transpose("y", "x", "band").values[..., :3]
    # uint8 imagery plots cleanly; float may need scaling
    if np.issubdtype(rgb.dtype, np.floating):
        scale = np.nanmax(rgb) or 1.0
        rgb = np.clip(rgb / scale, 0, 1)
    plt.imshow(rgb)
else:
    plt.imshow(np.asarray(arr).squeeze(), cmap="gray")
plt.title("OSM XYZ — stage-gate\n© OpenStreetMap contributors")
plt.axis("off")
plt.tight_layout()
plt.show()


## 1b. Same habit — EA DSP WCS (elevation)

`for_ea_dsp` enables AIMD under ceiling 10 + pressure breaker. Keep cold mosaics at/below the preset ceiling.


In [ ]:
cfg = ea_config()
da = create_array(cfg, ea_bbox(), CRS.EPSG_27700)
print("lazy:", da.shape)

t0 = time.perf_counter()
elevation = compute_with_policy(da)
wall = time.perf_counter() - t0
summarize(elevation, label="EA Lidar small mosaic", wall_s=wall)
print("mean elevation (m):", float(elevation.mean()))

plt.figure(figsize=(6, 4))
plt.imshow(elevation.values, cmap="terrain")
plt.colorbar(label="elevation (m)")
plt.title("EA Lidar DTM — stage-gate")
plt.axis("off")
plt.tight_layout()
plt.show()


## 2. Concurrency / politeness knobs — progress & timing

Re-run OSM (and optionally EA) with a **lower** call-local concurrency. Compare wall time and `tilearray_fetch_policy` (`max_concurrent` shared ceiling vs `call_max_concurrent`).

Set `OSM_MAX_CONCURRENT = 1` (or `EA_MAX_CONCURRENT = 4`) in the config cell and re-run this section — or override inline below.


In [ ]:
# Inline overrides for A/B without re-editing the top config cell
compare_osm_limits = [None, 1]  # None = preset (2); 1 = call-local serial feel

rows = []
for limit in compare_osm_limits:
    overrides = {}
    if limit is not None:
        overrides["max_concurrent_requests"] = limit
    cfg = osm_config(**overrides)
    da = create_array(cfg, osm_bbox(), CRS.EPSG_4326)
    t0 = time.perf_counter()
    out = compute_with_policy(da)
    wall = time.perf_counter() - t0
    fp = out.attrs.get("tilearray_fetch_policy", {})
    rows.append({
        "max_concurrent_requests": limit if limit is not None else "preset",
        "wall_s": round(wall, 2),
        "shape": out.shape,
        "shared_max_concurrent": fp.get("max_concurrent"),
        "call_max_concurrent": fp.get("call_max_concurrent"),
    })
    summarize(out, label=f"OSM limit={limit}", wall_s=wall)

print("comparison:")
for r in rows:
    pprint(r)


## 3. Soft-fail path — deliberate 404 mosaic

Default mosaics **hard-fail** after retries. Opt in with `soft_fail=True` (or `on_tile_error=soft_fail_exhausted_tiles`) at **plan time** on `create_array`.

This cell uses a tiny XYZ grid against a URL that always returns **404**, so you can inspect `attrs["tilearray_tile_errors"]` after `compute_with_policy` without relying on a flaky real host.

**Note:** band probing ignores soft-fail (shape discovery stays hard). We set `output_format=Format.GEOTIFF` here so the library uses the fixed single-band policy and skips the probe — otherwise a 404 URL would fail before the graph is built. Auth failures (401) still hard-abort under the soft-fail preset — do not “fix” credentials with soft-fail.


In [ ]:
# Deliberate always-404 template (no live OSM hammering).
# GEOTIFF → fixed band count → probe skipped (probe ignores soft-fail).
soft_cfg = XYZConfig.from_url(
    "https://httpbingo.org/status/404",
    zoom=1,
    output_format=Format.GEOTIFF,
    tile_size=64,
    max_concurrent_requests=2,
    fetch_retries=SOFT_FAIL_FETCH_RETRIES,
    adaptive_concurrency=False,
)

# Tiny bbox → a few tiles
soft_bbox = (0.0, 0.0, 0.01, 0.01)

da_soft = create_array(
    soft_cfg,
    soft_bbox,
    CRS.EPSG_4326,
    soft_fail=SOFT_FAIL,
    soft_fail_max_fraction=SOFT_FAIL_MAX_FRACTION,
)
print("armed tilearray_tile_errors (pre-compute):")
pprint(da_soft.attrs.get("tilearray_tile_errors"))

t0 = time.perf_counter()
try:
    soft_arr = compute_with_policy(da_soft)
    wall = time.perf_counter() - t0
    summarize(soft_arr, label="soft-fail 404 mosaic", wall_s=wall)
    print("finite fraction:", float(np.isfinite(soft_arr.values).mean()))
except Exception as exc:  # noqa: BLE001 — stage-gate: show hard-fail vs soft-fail
    wall = time.perf_counter() - t0
    print(f"raised after {wall:.2f}s: {type(exc).__name__}: {exc}")
    print("Tip: with SOFT_FAIL=False this should NetworkError; with True + high max_fraction, NaN fill.")


### Soft-fail off (control)

Flip to hard-fail on the same broken URL — expect `NetworkError` / mosaic abort.


In [ ]:
da_hard = create_array(
    soft_cfg,
    soft_bbox,
    CRS.EPSG_4326,
    soft_fail=False,
)
t0 = time.perf_counter()
try:
    compute_with_policy(da_hard)
    print("unexpected success")
except Exception as exc:  # noqa: BLE001
    print(f"expected hard fail after {time.perf_counter() - t0:.2f}s: {type(exc).__name__}: {exc}")


## 4. Optional — slightly larger AOI

Set `USE_LARGER_OSM_AOI = True` and/or `USE_LARGER_EA_AOI = True` in the config cell, then re-run sections 1–2. Stay polite; don't raise OSM past the preset.


## Notes while running

- **`load_array`:** same kwargs as `create_array` but defaults to `compute=True` (eager). Prefer lazy + `compute_with_policy` for multi-tile work.
- **Progress:** wall time above; for richer progress, pass `on_progress=...` to `create_array` (optional — not required for this gate).
- **Private hosts:** compose `params=` / `headers=` on the config (see config cell comments). No AuthClient.


## Checklist — what felt wrong? (fill in while running)

Edit this cell as you go. Anything awkward is a bug candidate unless proven otherwise.

- [ ] Happy path reads as prose: preset → `create_array` → `compute_with_policy`?
- [ ] Defaults feel safe / polite (OSM UA + limits; EA ceiling)?
- [ ] Progress / timing understandable? Anything silent or surprising?
- [ ] Attrs useful after compute (`tilearray_fetch_policy`, CRS/transform, soft-fail counters)?
- [ ] Soft-fail observability clear (`armed`, `soft_failed`, `examples`)?
- [ ] Plot / dtype / band layout surprising (palette PNG single-band, elevation nodata)?
- [ ] Error messages actionable when something fails?
- [ ] API names match what you expected (`max_concurrent_requests` vs mental model)?
- [ ] Performance feel OK for the small AOI? Larger AOI still polite?
- [ ] Anything you'd want before cutting **0.2.0**?

**Free notes:**

```
(your notes here)
```
